### Load packages

In [54]:
import pandas as pd
import os
import numpy as np
from datetime import datetime

### Define parameters and file info

In [55]:
# -----------------------------------------------------------------------
# Directory info
# -----------------------------------------------------------------------

github_data_directory = r"..\data\nonprofit_economy"
area990_data_directory = r"..\..\..\Documents\Work\IPS\Area 990\Data Sources\DAFs\csvs"

ntee_directory = r"..\..\..\AppData\Local\Programs\Python\Python310\Lib\site-packages\irsx\CSV"

output_directory = r"..\data\nonprofit_economy"

# -----------------------------------------------------------------------
# DAF-to-DAF data
# -----------------------------------------------------------------------

daf_basic_file = r"alldafs_basic_skedd_29mo_2023.csv"
daf_basic_path = os.path.join(area990_data_directory, daf_basic_file)
print(
    "DAF basic CSV file:",
    daf_basic_path,
    "EXISTS" if os.path.exists(daf_basic_path) else "MISSING"
)

# -----------------------------------------------------------------------
# Output files
# -----------------------------------------------------------------------

revenue_summary_file = r"DAF_revenue_summary_by_org_NTEE"
revenue_summary_path = os.path.join(output_directory, revenue_summary_file + ".csv")
print("Revenue Summary file:", revenue_summary_path)


DAF basic CSV file: ..\..\..\Documents\Work\IPS\Area 990\Data Sources\DAFs\csvs\alldafs_basic_skedd_29mo_2023.csv EXISTS
Revenue Summary file: ..\data\nonprofit_economy\DAF_revenue_summary_by_org_NTEE.csv


### Load in data

In [56]:
# Load DAF basic data

daf_basic_raw_df = pd.read_csv(daf_basic_path)

daf_basic_raw_df.head()

,Object ID,EIN,Name,Year of Tax Period End Date,Month of Tax Period End Date,Independent Audit Checkbox (Part XII Line 2b),Federal Audit Required Checkbox (Part XII Line 3a),Federal Audit Performed Checkbox (Part XII Line 3b),BMF Subsection Code,NTEE_COMBINED,...,Noncash Contributions Received,Percent Noncash Contributions,Investment Income (Part VIII Line 3(a)),Net Gain/Loss of Investment Sales (Part VIII Line 7d(a)),Unrealized Gain/Loss (Part XI Line 5),Investment Expenses (Part XI Line 7),DnrAdvsdFndsHldCnt,DnrAdvsdFndsCntrAmt,DnrAdvsdFndsGrntsAmt,DnrAdvsdFndsVlEOYAmt
0,202423099349303012,10391479,MAINE COMMUNITY FOUNDATION INC,2023,December,True,False,False,3.0,T310,...,10414630.0,0.204239,5418571.0,51118191.0,14865782.0,NaN,456.0,33164528.0,44959811.0,257527475.0
1,202402289349302920,10484310,MAINE INITIATIVESINC,2023,September,True,False,False,3.0,I20,...,88734.0,0.025796,67049.0,NaN,224324.0,NaN,12.0,339907.0,293500.0,742524.0
2,202400799349301215,10564355,THE FOUNDATION FOR ENHANCING COMMUNITIES,2023,December,True,False,False,3.0,P12,...,253703.0,0.062026,1873368.0,-274941.0,17399980.0,NaN,121.0,800529.0,3198098.0,16934064.0
3,202441279349301044,10589927,MINOT COMMUNITY FOUNDATION,2023,December,True,False,False,3.0,S20,...,2750.0,0.004653,1152211.0,3370.0,2963751.0,NaN,65.0,220260.0,236035.0,20433276.0
4,202422199349300742,10640027,MGM RESORTS FOUNDATION,2023,December,True,False,False,3.0,T30,...,NaN,0.000000,23840.0,NaN,NaN,NaN,18.0,465510.0,400901.0,256451.0


### Clean and format columns

In [57]:
# Clean & format columns in daf_basic data

daf_basic_clean_df = daf_basic_raw_df.copy()

# Rename EIN columns
daf_basic_clean_df.rename(
    columns={"DnrAdvsdFndsCntrAmt": "Contributions to DAFs", "DnrAdvsdFndsGrntsAmt": "Grants from DAFs"},
    inplace=True
)

# Convert integer columns to integer format
int_cols = ["EIN"]
daf_basic_clean_df[int_cols] = daf_basic_clean_df[int_cols].astype("Int64")

# Convert revenue columns to float
daf_basic_clean_df["Contributions to DAFs"] = daf_basic_clean_df["Contributions to DAFs"].astype(float)
daf_basic_clean_df["Grants from DAFs"] = daf_basic_clean_df["Grants from DAFs"].astype(float)

daf_basic_clean_df.head()

,Object ID,EIN,Name,Year of Tax Period End Date,Month of Tax Period End Date,Independent Audit Checkbox (Part XII Line 2b),Federal Audit Required Checkbox (Part XII Line 3a),Federal Audit Performed Checkbox (Part XII Line 3b),BMF Subsection Code,NTEE_COMBINED,...,Noncash Contributions Received,Percent Noncash Contributions,Investment Income (Part VIII Line 3(a)),Net Gain/Loss of Investment Sales (Part VIII Line 7d(a)),Unrealized Gain/Loss (Part XI Line 5),Investment Expenses (Part XI Line 7),DnrAdvsdFndsHldCnt,Contributions to DAFs,Grants from DAFs,DnrAdvsdFndsVlEOYAmt
0,202423099349303012,10391479,MAINE COMMUNITY FOUNDATION INC,2023,December,True,False,False,3.0,T310,...,10414630.0,0.204239,5418571.0,51118191.0,14865782.0,NaN,456.0,33164528.0,44959811.0,257527475.0
1,202402289349302920,10484310,MAINE INITIATIVESINC,2023,September,True,False,False,3.0,I20,...,88734.0,0.025796,67049.0,NaN,224324.0,NaN,12.0,339907.0,293500.0,742524.0
2,202400799349301215,10564355,THE FOUNDATION FOR ENHANCING COMMUNITIES,2023,December,True,False,False,3.0,P12,...,253703.0,0.062026,1873368.0,-274941.0,17399980.0,NaN,121.0,800529.0,3198098.0,16934064.0
3,202441279349301044,10589927,MINOT COMMUNITY FOUNDATION,2023,December,True,False,False,3.0,S20,...,2750.0,0.004653,1152211.0,3370.0,2963751.0,NaN,65.0,220260.0,236035.0,20433276.0
4,202422199349300742,10640027,MGM RESORTS FOUNDATION,2023,December,True,False,False,3.0,T30,...,NaN,0.000000,23840.0,NaN,NaN,NaN,18.0,465510.0,400901.0,256451.0


### Create summaries by NTEE code

In [ ]:
# Function to calculate Shonni's summary groupings on 

def shonni_groups(row, first_digit_ntee_code, full_ntee_code):

    code = row[first_digit_ntee_code]
    combined = str(row[full_ntee_code]).upper() if pd.notna(row[full_ntee_code]) else ""
    
    if pd.isna(code):
        return None
    
    code = str(code).upper()

    if combined.startswith(("T30", "T31", "T32")): # private foundations, community foundations, and corporate foundations
        return "Foundations"
    
    if code == "A":
        return "Arts, Culture, Humanities"
    
    elif code == "B":
        if combined.startswith(("B40", "B41", "B43", "B50")):
            return "Colleges and Universities"
        else:
            return "Other Education"
    
    elif code in ["C", "D"]:
        return "Environment and Animals"
    
    elif code in ["E", "F", "G", "H"]:
        if combined.startswith(("E20", "E21", "E22", "E24", "E90", "E91", "E92")):
            return "Hospitals and Nursing Homes"
        else:
            return "Other Health"
    
    elif code in ["I", "J", "K", "L", "M", "N", "O", "P"]:
        return "Human Services"
    
    elif code == "Q":
        return "International, Foreign Affairs"
    
    elif code in ["R", "S", "T", "U", "V", "W"]:
        return "Public, Societal Benefit"
    
    elif code == "X":
        return "Religion Related"
    

    elif code in ["Y", "Z"]:
        return "Unknown, Unclassified"
    
    else:
        return None

In [59]:
# Apply Shonni groupings to daf-to-daf grants by RECIPIENT NTEE code

daf_basic_grouped_df = daf_basic_clean_df.copy()

# Create one-digit NTEE code columm
daf_basic_grouped_df["NTEE_FIRST_DIGIT"] = daf_basic_grouped_df["NTEE_COMBINED"].str[0]

daf_basic_grouped_df["NTEE_ACTIVITY"] = daf_basic_grouped_df.apply(
    shonni_groups,
    axis=1,
    args=("NTEE_FIRST_DIGIT", "NTEE_COMBINED")
)

daf_basic_grouped_df.head()

,Object ID,EIN,Name,Year of Tax Period End Date,Month of Tax Period End Date,Independent Audit Checkbox (Part XII Line 2b),Federal Audit Required Checkbox (Part XII Line 3a),Federal Audit Performed Checkbox (Part XII Line 3b),BMF Subsection Code,NTEE_COMBINED,...,Investment Income (Part VIII Line 3(a)),Net Gain/Loss of Investment Sales (Part VIII Line 7d(a)),Unrealized Gain/Loss (Part XI Line 5),Investment Expenses (Part XI Line 7),DnrAdvsdFndsHldCnt,Contributions to DAFs,Grants from DAFs,DnrAdvsdFndsVlEOYAmt,NTEE_FIRST_DIGIT,NTEE_ACTIVITY
0,202423099349303012,10391479,MAINE COMMUNITY FOUNDATION INC,2023,December,True,False,False,3.0,T310,...,5418571.0,51118191.0,14865782.0,NaN,456.0,33164528.0,44959811.0,257527475.0,T,"Public, Societal Benefit"
1,202402289349302920,10484310,MAINE INITIATIVESINC,2023,September,True,False,False,3.0,I20,...,67049.0,NaN,224324.0,NaN,12.0,339907.0,293500.0,742524.0,I,Human Services
2,202400799349301215,10564355,THE FOUNDATION FOR ENHANCING COMMUNITIES,2023,December,True,False,False,3.0,P12,...,1873368.0,-274941.0,17399980.0,NaN,121.0,800529.0,3198098.0,16934064.0,P,Human Services
3,202441279349301044,10589927,MINOT COMMUNITY FOUNDATION,2023,December,True,False,False,3.0,S20,...,1152211.0,3370.0,2963751.0,NaN,65.0,220260.0,236035.0,20433276.0,S,"Public, Societal Benefit"
4,202422199349300742,10640027,MGM RESORTS FOUNDATION,2023,December,True,False,False,3.0,T30,...,23840.0,NaN,NaN,NaN,18.0,465510.0,400901.0,256451.0,T,"Public, Societal Benefit"


In [ ]:
# Define activity area sort order

activity_order = ["Arts, Culture, Humanities", "Other Education", "Colleges and Universities", "Environment and Animals", "Other Health", 
         "Hospitals and Nursing Homes", "Human Services", "International, Foreign Affairs", "Public, Societal Benefit",
         "Foundations", "Religion Related", "Unknown, Unclassified"]


In [61]:
# Create summary of DAF basic data

daf_basic_summary_df = daf_basic_grouped_df.copy()

daf_basic_summary_df["NTEE_ACTIVITY"] = daf_basic_summary_df.apply(
    shonni_groups,
    axis=1,
    args=("NTEE_FIRST_DIGIT", "NTEE_COMBINED")
)

daf_basic_summary = (
    daf_basic_summary_df
    .groupby("NTEE_ACTIVITY", as_index=False)
    .agg(
        num_dafs=("EIN", "count"),
        daf_contribs=("Contributions to DAFs", "sum"),
        daf_grants=("Grants from DAFs", "sum")
    )
)

daf_basic_summary["NTEE_ACTIVITY"] = pd.Categorical(
    daf_basic_summary["NTEE_ACTIVITY"],
    categories=activity_order,
    ordered=True
)

daf_basic_summary = daf_basic_summary.sort_values("NTEE_ACTIVITY")

daf_basic_summary

,NTEE_ACTIVITY,num_dafs,daf_contribs,daf_grants
0,"Arts, Culture, Humanities",41,3.555447e+06,5.363987e+06
6,Other Education,142,8.639387e+08,3.749055e+08
1,Colleges and Universities,32,1.055058e+08,1.802652e+08
2,Environment and Animals,31,1.414752e+07,1.001469e+07
7,Other Health,67,1.828330e+07,8.447245e+06
3,Hospitals and Nursing Homes,16,9.016015e+06,4.336445e+06
4,Human Services,149,5.081036e+08,4.263978e+08
5,"International, Foreign Affairs",34,6.699852e+07,6.210240e+07
8,"Public, Societal Benefit",1063,6.229238e+10,5.213562e+10
9,Religion Related,75,1.102570e+09,8.558889e+08


### Output results

In [63]:
# Output the recipient summary table to csv

# Archive old output file if present

if os.path.exists(revenue_summary_path):

    # Get modification time of existing file
    old_timestamp = datetime.fromtimestamp(
        os.path.getmtime(revenue_summary_path)
    ).strftime("%Y%m%d_%H%M%S")

    archived_filename = (
        f"{revenue_summary_file}_archived_{old_timestamp}.csv"
    )

    archived_path = os.path.join(
        output_directory,
        archived_filename
    )

    os.rename(
        revenue_summary_path,
        archived_path
    )

    print(
        "Archived existing file:",
        archived_path
    )

daf_basic_summary.to_csv(revenue_summary_path, index=False)

Archived existing file: ..\data\nonprofit_economy\DAF_revenue_summary_by_org_NTEE_archived_20260701_142737.csv
